# Evaluation Notebook

This notebook outsources the logic from `src/evaluation/evaluation.py` into an interactive workflow.

Before running all cells:
1. Start the API server (default: `http://127.0.0.1:3100`).
2. Ensure the database is populated and actor classification data exists.

In [ ]:
import asyncio
import os
import sys
from typing import List

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

In [ ]:
# Ensure imports from src/ resolve independent of notebook working directory
cwd = os.path.abspath(os.getcwd())

# Try common locations first (repo root, api root, notebooks root)
roots_to_probe = [
    cwd,
    os.path.dirname(cwd),
    os.path.dirname(os.path.dirname(cwd)),
]

candidate_paths = []
for root in roots_to_probe:
    candidate_paths.extend(
        [
            os.path.join(root, 'src'),
            os.path.join(root, 'movie-actor-ranking-api', 'src'),
            os.path.join(root, 'notebooks', '..', 'src'),
        ]
    )

# Normalize and de-duplicate while preserving order
normalized_candidates = []
seen = set()
for path in candidate_paths:
    normalized = os.path.abspath(path)
    if normalized not in seen:
        normalized_candidates.append(normalized)
        seen.add(normalized)

PROJECT_SRC = None
for candidate in normalized_candidates:
    if os.path.isfile(os.path.join(candidate, 'config.py')) and os.path.isdir(
        os.path.join(candidate, 'db')
    ):
        PROJECT_SRC = candidate
        break

# Fallback: shallow search for any src folder that matches expected structure
if PROJECT_SRC is None:
    for root in roots_to_probe:
        try:
            for entry in os.listdir(root):
                maybe_src = os.path.join(root, entry, 'src')
                if os.path.isfile(os.path.join(maybe_src, 'config.py')) and os.path.isdir(
                    os.path.join(maybe_src, 'db')
                ):
                    PROJECT_SRC = os.path.abspath(maybe_src)
                    break
        except FileNotFoundError:
            continue

        if PROJECT_SRC is not None:
            break

if PROJECT_SRC is None:
    raise RuntimeError(
        f'Could not locate project src directory with config.py and db/. cwd={cwd}'
    )

if PROJECT_SRC not in sys.path:
    sys.path.insert(0, PROJECT_SRC)

from config import (
    GROUND_DATASET_FILE_PATH,
    EVAL_MEASURES_IMAGE_PATH,
    EVAL_MEASURES_CSV_PATH,
)
from db.actor import get_actors_by_names
from db.models import Actor

In [ ]:
# Config
base_url = 'http://127.0.0.1:3100'
vector_space_url = f'{base_url}/search/classifier/actor'

queries = [
    'romantic scenes',
    'action filled dramas',
    'funniest actors ever',
    'heartbreaking roles',
    'saddest movie performances',
    'action-packed movie stars',
]

In [ ]:
def evaluate_search_model(relevant_docs: List[int], retrieved_docs: List[int]) -> tuple:
    relevant_retrieved_docs = set(relevant_docs).intersection(set(retrieved_docs))

    true_positives = len(relevant_retrieved_docs)
    false_positives = len(retrieved_docs) - true_positives
    false_negatives = len(relevant_docs) - true_positives

    recall = true_positives / (true_positives + false_negatives) if relevant_docs else 0
    precision = true_positives / (true_positives + false_positives) if retrieved_docs else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return recall, precision, f1


def call_vector_space_api(query: str) -> List[Actor]:
    query_string = '+'.join(query.split())
    url = f'{vector_space_url}?q={query_string}'
    response = requests.get(url, timeout=30)
    response.raise_for_status()

    actors_json = response.json()
    return [Actor(**actor) for actor in actors_json]


async def get_relevant_actors(query: str, ground_truth_df: pd.DataFrame) -> List[int]:
    df_query = ground_truth_df[query]
    db_actors = await get_actors_by_names(df_query.values.tolist())
    relevant_docs = [actor.id for actor in db_actors]
    relevant_docs.sort()
    return relevant_docs


def plot_evaluation_results(results: pd.DataFrame):
    axes = results.set_index('query')[
        ['vector_space_recall', 'vector_space_precision', 'vector_space_f1']
    ].plot(kind='bar', subplots=True, layout=(3, 1), legend=False)

    axes = np.ravel(axes)

    for i, ax in enumerate(axes):
        ax.set_ylim([0, 1])
        ax.set_ylabel('Score')

        if i == len(axes) - 1:
            ax.set_xticklabels(results['query'], rotation=45, horizontalalignment='right')
            ax.set_xlabel('')
        else:
            ax.set_xticklabels([])

    plt.tight_layout()
    plt.savefig(EVAL_MEASURES_IMAGE_PATH)
    plt.show()


async def calculate_differences():
    df = pd.read_csv(GROUND_DATASET_FILE_PATH, sep=';', index_col='rank')

    for query in queries:
        print(f'Query: {query}')
        df_query = df[query]
        db_actors = await get_actors_by_names(df_query.values.tolist())

        diff = set(df_query.values.tolist()) - set([actor.name for actor in db_actors])
        print('original: ', len(df_query))
        print('db: ', len(db_actors))
        print(diff)


async def run_evaluation() -> pd.DataFrame:
    print('Evaluate Vector Space Models')
    print('===========================================')

    ground_truth_df = pd.read_csv(GROUND_DATASET_FILE_PATH, sep=';', index_col='rank')
    results = []

    for query in queries:
        print(f'Query: {query}')

        relevant_actor_ids = await get_relevant_actors(query, ground_truth_df)
        print('Relevant Actors: ', relevant_actor_ids)

        vector_space_actors = call_vector_space_api(query)
        vector_space_actors_ids = [actor.id for actor in vector_space_actors]
        vector_space_actors_ids.sort()
        print('Vector Space Actors IDs: ', vector_space_actors_ids)

        vector_space_recall, vector_space_precision, vector_space_f1 = evaluate_search_model(
            relevant_actor_ids, vector_space_actors_ids
        )

        results.append({
            'query': query,
            'vector_space_recall': vector_space_recall,
            'vector_space_precision': vector_space_precision,
            'vector_space_f1': vector_space_f1,
        })

    df = pd.DataFrame(results)
    df.describe().to_csv(EVAL_MEASURES_CSV_PATH)
    plot_evaluation_results(df)

    return df

In [ ]:
# Optional debugging helper:
# await calculate_differences()

results_df = await run_evaluation()
results_df